# Covering with affine approximations and CICO cryptanalysis

We need `ortools` for solving SetCover problem + optimizing probabilistic coverage.

In [1]:
from sage.all import *
from binteger import Bin
import time
assert 2^1 == 3  # python mode!!!

In [2]:
%pip install ortools tqdm binteger


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
STATUS = {0: "optimal", 1: "suboptimal", 2: "unsolvable"}

In [4]:
import subprocess, ast
GREEDY_EXTENSION_PATH = "./greedy_extension/target/release/greedy_extention"

SKINNY_8 = (101, 76, 106, 66, 75, 99, 67, 107, 85, 117, 90, 122, 83, 115, 91, 123, 53, 140, 58, 129, 137, 51, 128, 59, 149, 37, 152, 42, 144, 35, 153, 43, 229, 204, 232, 193, 201, 224, 192, 233, 213, 245, 216, 248, 208, 240, 217, 249, 165, 28, 168, 18, 27, 160, 19, 169, 5, 181, 10, 184, 3, 176, 11, 185, 50, 136, 60, 133, 141, 52, 132, 61, 145, 34, 156, 44, 148, 36, 157, 45, 98, 74, 108, 69, 77, 100, 68, 109, 82, 114, 92, 124, 84, 116, 93, 125, 161, 26, 172, 21, 29, 164, 20, 173, 2, 177, 12, 188, 4, 180, 13, 189, 225, 200, 236, 197, 205, 228, 196, 237, 209, 241, 220, 252, 212, 244, 221, 253, 54, 142, 56, 130, 139, 48, 131, 57, 150, 38, 154, 40, 147, 32, 155, 41, 102, 78, 104, 65, 73, 96, 64, 105, 86, 118, 88, 120, 80, 112, 89, 121, 166, 30, 170, 17, 25, 163, 16, 171, 6, 182, 8, 186, 0, 179, 9, 187, 230, 206, 234, 194, 203, 227, 195, 235, 214, 246, 218, 250, 211, 243, 219, 251, 49, 138, 62, 134, 143, 55, 135, 63, 146, 33, 158, 46, 151, 39, 159, 47, 97, 72, 110, 70, 79, 103, 71, 111, 81, 113, 94, 126, 87, 119, 95, 127, 162, 24, 174, 22, 31, 167, 23, 175, 1, 178, 14, 190, 7, 183, 15, 191, 226, 202, 238, 198, 207, 231, 199, 239, 210, 242, 222, 254, 215, 247, 223, 255)

def run_greedy_extension(S, n, m, itr=1):
    proc = subprocess.Popen([GREEDY_EXTENSION_PATH, str(itr)], stdin=subprocess.PIPE, stdout=subprocess.PIPE)
    data = f"%d %d\n%s" % (n, m, " ".join(map(str, S)))
    out, err = proc.communicate(data.encode())
    out = out.decode()
    out = out.splitlines()[-2].split(" [")[-1].strip().strip("]")
    return ast.literal_eval(out)

## Collect many approximations

First, we collect 2000 approximations from random executions of the greedy algorithm. Randomization is assured because of the initial point choices in the algorithm.

In [5]:
import random
from collections import Counter, defaultdict
from tqdm import tqdm

n = m = 8
S = SKINNY_8
Si = [list(S).index(x) for x in range(2**n)]

approx = []
for _ in tqdm(range(2000)):
    xs = run_greedy_extension(S, n=n, m=m, itr=1)
    approx.append(xs)

assert len(set(sum(approx, ()))) == 2**n, "not all inputs covered"

len(approx), Counter(map(len, approx))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:04<00:00, 486.23it/s]


(2000,
 Counter({45: 496,
          44: 299,
          47: 292,
          42: 219,
          43: 137,
          40: 117,
          39: 116,
          41: 108,
          38: 58,
          36: 41,
          37: 38,
          46: 30,
          35: 25,
          33: 11,
          34: 7,
          31: 3,
          32: 2,
          29: 1}))

In [6]:
sum(1 for v in approx if len(v) >= 40)

1698

For each input, store which sets cover it.

In [7]:
k = len(approx)
sets_for_x = defaultdict(list)
for i, st in enumerate(approx):
    for x in st:
        sets_for_x[x].append(i)

## Method 1: deterministic coverage (via SetCover)

In this method, we will just try to choose a ~smallest subset of those 2000 approximations that would cover all the input. In the attack, we will treat it as a partition and will enumerate each of the candidate for each S-box.

Finding the optimally small subset is the NP-hard SetCover problem. We will just use a heuristic solution by an ILP solver. Note that even the optimal solution is unlikely to be much smaller.

In [8]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver('SCIP')

take = [solver.BoolVar(f'take_{i}') for i in range(k)]

objective = solver.Objective()
for i in range(k):
    objective.SetCoefficient(take[i], 1)
objective.SetMinimization()

for x in range(2**n):
    solver.Add(solver.Sum(take[i] for i in sets_for_x[x]) >= 1)
n, k, solver

(8,
 2000,
 <ortools.linear_solver.pywraplp.Solver; proxy of <Swig Object of type 'operations_research::MPSolver *' at 0x7cd215779e30> >)

In [9]:
solver.SetTimeLimit(10_000) # in milliseconds
# 0: optimal solution
# 1: solution found (suboptimal?)
# else: not ready yet
status = solver.Solve()
print("Status", status, ":", STATUS.get(status, "?"))
if status in (0, 1):
    print("objective", solver.Objective().Value())
    sol = []
    total = 0
    num = 0
    for i in range(k):
        if take[i].solution_value() > 0:
            print("take", i, approx[i])
            sol.append(i)
            total += len(approx[i])
            num += 1
    print()
    print("objective", solver.Objective().Value(), "approximations")
    print("overhead (number of redundant input covers)", total - 2**n)
    print("avg approximation size", total / num,
          "min max", min(len(approx[i]) for i in sol), max(len(approx[i]) for i in sol))
    print("-------------")
    print("(!) avg effective approximation size", 2**n / num)
    print("-------------")
    print()
print("indexes of selected sets:", sol)

Status 1 : suboptimal
objective 12.0
take 8 (20, 22, 26, 28, 30, 36, 38, 42, 43, 44, 45, 46, 47, 68, 70, 74, 76, 78, 84, 86, 90, 91, 92, 93, 94, 95, 116, 118, 122, 123, 124, 125, 126, 127, 196, 202, 206, 212, 218, 219, 222, 223, 244, 250, 251, 254, 255)
take 18 (26, 28, 30, 37, 39, 43, 45, 47, 53, 55, 59, 61, 63, 69, 71, 74, 75, 76, 77, 78, 79, 101, 103, 106, 107, 108, 109, 110, 111, 117, 119, 123, 125, 127, 199, 202, 203, 206, 207, 231, 234, 235, 238, 239, 247, 251, 255)
take 19 (7, 11, 15, 23, 26, 27, 28, 30, 31, 53, 55, 58, 59, 61, 62, 63, 69, 71, 75, 77, 79, 85, 87, 91, 93, 95, 106, 108, 110, 133, 135, 138, 139, 141, 142, 143, 149, 151, 155, 157, 159, 167, 170, 171, 172, 174, 175)
take 20 (0, 2, 9, 11, 13, 16, 18, 24, 25, 27, 29, 48, 56, 57, 58, 60, 128, 136, 137, 141, 144, 153, 157, 160, 168, 169, 172, 192, 194, 201, 202, 203, 204, 205, 208, 210, 217, 219, 221, 226, 232, 234, 235, 236, 237)
take 100 (1, 4, 8, 10, 16, 17, 18, 24, 48, 49, 52, 56, 58, 66, 67, 68, 69, 74, 81, 83, 84, 

We see that we can cover the input space with 11 approximations, yielding effective approximation size 256/11=23.27 points. This is twice lower than the best / most approximations in the pool!

For $t$ S-boxes, we would need $11^t$ attempts to be able to catch any possible solution.

**Note:** on average, we would need $11^t/2$ attempts: do not confuse with $(11/2)^t$, even though for one S-box the expectation *is* $11/2$!

## Method 2: probabilistic (almost-)uniform coverage

### Option 1: almost-uniform (optimize average)

Here, we set a uniformity bound and find a distribution of the approximations achieving a close-to-uniform probability for every input $x$, maximizing the average probability at the same time.

In [10]:
from ortools.linear_solver import pywraplp

BOUND = 2**-8.0
print("BOUND", BOUND, "=", BOUND * 2**n, "/2^n")

solver = pywraplp.Solver.CreateSolver('SCIP')

wt = [
    solver.Var(0, 1, False, "wt%d" % 0)
    for i in range(k)
]
solver.Add(sum(wt) == 1.0)
           
vec = []
for x in range(2**n):
    weights_for_x = [wt[i] for i in sets_for_x[x]]
    vec.append(solver.Sum(weights_for_x))

# faster but less precise (compare to min prob.)
#min_i = randrange(2**n) # guess the minimum element
#inds = [i for i in range(2**n) if i != min_i]
#print("min_i", min_i)
# for i in inds:
#     solver.Add(vec[i] - vec[min_i] >= 0)
#     solver.Add(vec[i] - vec[min_i] <= BOUND)

print("Generating equations (slow interface)...")
# slower but more precise (compare to avg prob.)
# slow because of the interface...
avg = solver.Sum(vec) / len(vec)
for i in tqdm(range(2**n)):
    solver.Add(vec[i] - avg >= -BOUND)
    solver.Add(vec[i] - avg <= BOUND)

solver.Maximize(sum(len(S) * wt[i] for i, S in enumerate(approx)))

solver.SetTimeLimit(5_000) # in milliseconds

print("Solving...")
# 0: optimal solution
# 1: solution found (suboptimal?)
# else: not ready yet
status = solver.Solve()
print("Status", status, ":", STATUS.get(status, "?"))
if status == 2:
    print("NO SOLUTION! Need more approximations ot reduce the bound.")
if status in (0, 1):
    print("objective", solver.Objective().Value(), "= expected approx. size")
    weights = [wt[i].solution_value() for i in range(k)]
    probs = [v.solution_value() for v in vec]
    print("min prob", min(probs), "max prob", max(probs))
    print("avg prob", sum(probs) / len(probs))
    dev = (max(probs) - min(probs)) / 2
    print("max deviation", dev, "= 1 /", 1/dev)

BOUND 0.00390625 = 1.0 /2^n
Generating equations (slow interface)...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:21<00:00, 11.71it/s]


Solving...
Status 0 : optimal
objective 43.77600557808191 = expected approx. size
min prob 0.16709377178938023 max prob 0.1749062717893841
avg prob 0.17100002178938245
max deviation 0.003906250000001929 = 1 / 255.99999999987358


The effective approximation size is 43.81, very close to the maximum 47!

### Option 2: lower bounded (optimize minimum)

The alternative option is to maximize the minimum probability over the inputs. This provides a robust probability of success working for all possible solutions of the problem.

In [11]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver('SCIP')

bound = solver.Var(0, 1, False, "bound")
wt = [
    solver.Var(0, 1, False, "wt%d" % i)
    for i in range(k)
]
solver.Add(sum(wt) == 1.0)
           
vec = []
for x in range(2**n):
    weights_for_x = [wt[i] for i in sets_for_x[x]]
    vec.append(solver.Sum(weights_for_x))

# lower bound ...
for i in range(2**n):
    solver.Add(vec[i] >= bound)

# + maximize
solver.Maximize(bound)

solver.SetTimeLimit(5_000) # in milliseconds

# 0: optimal solution
# 1: solution found (suboptimal?)
# else: not ready yet
status = solver.Solve()
print("Status", status, ":", STATUS.get(status, "?"))
if status == 2:
    print("NO SOLUTION! Need more approximations ?")

if status in (0, 1):
    obj = solver.Objective().Value()
    print("objective", obj, "* 2^n = %f = expected approx. size" % (2**n*obj))
    weights = [wt[i].solution_value() for i in range(k)]
    probs = [v.solution_value() for v in vec]
    print("min prob", min(probs), "max prob", max(probs))
    print("avg prob", sum(probs) / len(probs))
    dev = (max(probs) - min(probs)) / 2
    print("max deviation", dev, "= 1 /", 1/dev)

Status 0 : optimal
objective 0.16981111524934264 * 2^n = 43.471646 = expected approx. size
min prob 0.16981111524934153 max prob 0.18554508038024337
avg prob 0.16996597033199018
max deviation 0.007866982565450917 = 1 / 127.11353961703897


In [12]:
len([p for p in sol if p])

12

The effective approximation size is 43.5, slightly lower than the average-maximizing approach, yet with stronger guarantee. It also uses only 243 of the 2000 approximations (with varying weights).

## Application to CICO with SKINNY

We show how to use linearization to solve a CICO problem.

In [37]:
def AffineSpan(l):
    if len(l) <= 1:
        return l
    off = l[0]
    res = {0}
    for v in l[1:]:
        v ^= off
        if v not in res:
            res |= {u ^ v for u in res}
    res = [u ^ off for u in res]
    return res
    
def approximation_from_inputs(xs, S, n, m):
    graph = [(S[x] << n) | x for x in xs]
    asp = AffineSpan(graph)
    if len(asp) != 2**n:
        raise ValueError("approximation is not maximal")
    A = [None] * 2**n
    mask = (1 << n) - 1
    for yx in asp:
        y = yx >> n
        x = yx & mask
        A[x] = y
    assert None not in A
    return A

def xs_to_mat_const(xs, S, n, m):
    aff = approximation_from_inputs(xs, S, n=n, m=m)
    
    const = Bin(aff[0], n).vector
    lin = [y ^ aff[0] for y in aff]
    
    mat = matrix(GF(2), [
        Bin(lin[2**(n-1-i)], n) for i in range(n)
    ]).transpose()

    # testing
    if 1:
        for x in Bin.iter(n):
            y = mat * vector(x.vector) + const
            assert aff[x.int] == Bin(y).int, x
    return mat, const

def P(xs):
    state = list(xs)
    for i in range(NR):
        state = [S[x] for x in state]
        if i < NR - 1:
            # apply linear layer
            state = Bin.concat(*state, n=n).vector
            state = LIN * state
            state = [v.int for v in Bin(state).split(n=n)]
    return state

### Option 1: instance randomization

Here, we will try to linearize all S-boxes with the same best approximation. In case of failure, we just ask another CICO instance.

Convert the best affine approximation to matrix + constant:

In [14]:
for xs in approx:
    if len(xs) == 47:
        break

best_approx = xs
best_approx_mat, best_approx_const = xs_to_mat_const(best_approx, S, n=n, m=n)
best_approx_mat, best_approx_const

([0 1 1 1 0 0 0 0]
 [0 1 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 1]
 [0 0 0 0 1 0 0 0]
 [0 0 0 0 1 0 1 0]
 [0 1 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0]
 [0 0 0 0 1 1 1 0],
 (0, 1, 0, 0, 1, 0, 0, 0))

Define the permutation:

In [20]:
#c = r = 2 # 16 + 16 bits
c = r = 3 # 24 + 24 bits
m = c + r

NR = 3
LIN1 = LIN = GL(m*n, GF(2)).random_element().matrix()
c, r, LIN

(3, 3, 48 x 48 dense matrix over Finite Field of size 2)

In [38]:
P([0] * (c+r)), P([1] + [0] * (c+r-1))

([102, 246, 243, 238, 35, 198], [210, 92, 70, 154, 90, 39])

We change the instance by asking different plaintext values fixed (instead of zeroes). Outputs are always fixed to zeroes.

In [39]:
tries = 0
wins = 0
solutions = []

Expected success probability:

In [40]:
print("Expected success rate: %.2f" % math.log( (len(best_approx) / 2**n)**m, 2))

Expected success rate: -14.67


Note: we do not ensure the existence of the solution, so an instance may have 0 solutions, or 1, or more. Since the case of many solutions improve the success rate, we obtain the same expected success rate:

$\sum_{k=0}^\infty \frac{k}{k!\cdot e} \approx 1$

In [44]:
import time
t_start = time.time()
LIN = LIN1
for itr in tqdm(range(2**20)):
    instance = Bin.random(n*c)
    instance = list(instance.split(c))

    tries += 1
    # if tries % 1000 == 0:
    #     print(tries, ct, "time per iter:", (time.time() - t_start) / tries, "seconds")
    
    xs = PolynomialRing(GF(2), names=["x%d" % i for i in range(n*c)]).gens()
    
    # S layer: free
    state = (
        [Bin(S[v.int], n).vector for v in instance]
        + [xs[i:i+n] for i in range(0, len(xs), n)]
    )
    
    # L layer
    state = sum(map(list, state), [])
    state = LIN * vector(state)
    state = [state[i:i+n] for i in range(0, len(state), n)]

    # S layer (middle)
    for i in range(len(state)):
        state[i] = best_approx_mat * state[i] + best_approx_const

    # L layer
    state = sum(map(list, state), [])
    state = LIN * vector(state)
    #state = [state[i:i+n] for i in range(0, len(state), n)]

    # S layer: free
    output = state[:c*n]
    target = vector(GF(2), Bin(Si[0], n).list * c)
    seq = Sequence(output - target)
    
    mat, monos = seq.coefficients_monomials()
    if mat.ncols() != c*n+1:
        continue
        
    tar = mat.column(-1)
    mat = mat[:,:-1]
    try:
        base_sol = mat.solve_right(tar)
    except:
        continue
    for zero in mat.right_kernel():
        sol = base_sol + zero
        pt = [v.int for v in instance] + [Si[v.int] for v in Bin(sol).split(r)]
        ct = P(pt)
        if ct[:c] == [0]*c:
            wins += 1
            solutions.append((instance, pt, ct))
            if wins == 1:
                print("Solved (1st time):", pt, ct, ":",
                      "ratio %.5f" % (wins / tries),
                      "success probability %.2f" % math.log(wins / tries, 2)
                )

  8%|████████████████▏                                                                                                                                                                              | 88846/1048576 [50:52<9:09:31, 29.11it/s]


KeyboardInterrupt: 

In [ ]:
print("Wins / tries ratio %.5f" % (wins / tries),
      "= success probability %.2f" % math.log(wins / tries, 2)
)
print("Example:", solutions[0])

solutions1 = solutions

### Option 2: probabilistic coverage

Here, we will restrict ourselves to a single CICO instance, and will use the probabilistic coverage method to find solutions.

In [ ]:
from binteger import Bin
mats = []
for xs in tqdm(approx):
    mats.append(xs_to_mat_const(xs, S, n=n, m=n))

Define the permutation:

In [32]:
#c = r = 2 # 16 + 16 bits
c = r = 3 # 24 + 24 bits
m = c + r

NR = 3
LIN2 = LIN = GL(m*n, GF(2)).random_element().matrix()
c, r, LIN

(3, 3, 48 x 48 dense matrix over Finite Field of size 2)

In [ ]:
P([0] * (c+r)), P([1] + [0] * (c+r-1))

We change the instance by asking different plaintext values fixed (instead of zeroes). Outputs are always fixed to zeroes.

In [ ]:
tries = 0
wins = 0
solutions = []

Expected success probability:

In [ ]:
g = sum(weights[i] * len(approx[i]) for i in range(len(approx)))
print("Expected success rate: %.2f" % math.log( (g / 2**n)**m, 2))

Note: solvability depends on LIN, so may need to rerandomize if no solution exists.

In [ ]:
t_start = time.time()
LIN = LIN2
for itr in tqdm(range(2**20)):
    tries += 1
    # if tries % 1000 == 0:
    #     print(tries, ct, apps[1], "per iter", (time.time() - t_start) / tries, "wins", wins)

    # guess approximations
    # according to the distribution
    apps = []
    inds = list(range(len(approx)))
    for i in range(NR):
        row = []
        for j in range(m):
            app_i, = random.choices(inds, weights=weights)
            row.append(app_i)
        apps.append(row)
    
    xs = PolynomialRing(GF(2), names=["x%d" % i for i in range(n*c)]).gens()
    
    # S layer: free
    state = [Bin(S[0], n).vector] * c + [xs[i:i+n] for i in range(0, len(xs), n)]

    # L layer
    state = sum(map(list, state), [])
    state = LIN * vector(state)
    state = [state[i:i+n] for i in range(0, len(state), n)]

    # S layer (middle)
    for i in range(len(state)):
        app_i = apps[1][i]  # middle round approx.
        mat, const = mats[app_i]
        state[i] = mat * state[i] + const

    # L layer
    state = sum(map(list, state), [])
    state = LIN * vector(state)
    #state = [state[i:i+n] for i in range(0, len(state), n)]

    # S layer: free
    output = state[:c*n]
    target = vector(GF(2), Bin(Si[0], n).list * c)
    seq = Sequence(output - target)
    
    mat, monos = seq.coefficients_monomials()
    if mat.ncols() != c*n+1:
        continue
    
    tar = mat.column(-1)
    mat = mat[:,:-1]
    try:
        base_sol = mat.solve_right(tar)
    except:
        continue
    
    for zero in mat.right_kernel():
        sol = base_sol + zero
        pt = [0] * c + [Si[v.int] for v in Bin(sol).split(r)]
        ct = P(pt)
        if ct[:c] == [0]*c:
            wins += 1
            solutions.append((instance, pt, ct))
            if wins == 1:
                print("Solved (1st time):", pt, ct, ":",
                      "ratio %.5f" % (wins / tries),
                      "success probability %.2f" % math.log(wins / tries, 2)
                )

In [ ]:
print("Wins / tries ratio %.5f" % (wins / tries),
      "= success probability %.2f" % math.log(wins / tries, 2)
)
print("Example:", solutions[0])

solutions2 = solutions

Note: linear algebra is done using composition of matrices and symbolic variables; that's why it is slow. A simple optimization would be to interpolate the matrix and then solve it.